# NCAA March Madness Meta-Stacking (v5)

This notebook implements a sophisticated Stacking architecture:
1. **Base Models**: XGBoost, LightGBM, and Neural Network (MLP).
2. **Meta-Learner**: Logistic Regression trained on Out-of-Fold (OOF) predictions.
3. **Features**: Includes Elo ratings, SOS, Trends, and Win Percentage.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.metrics import log_loss
from sklearn.preprocessing import StandardScaler
import os

DATA_PATH = './march-machine-learning-mania-2026/'
team_features = pd.read_csv('all_team_features_v3.csv')
m_tourney = pd.read_csv(os.path.join(DATA_PATH, 'MNCAATourneyDetailedResults.csv'))
w_tourney = pd.read_csv(os.path.join(DATA_PATH, 'WNCAATourneyDetailedResults.csv'))
tourney_results = pd.concat([m_tourney, w_tourney])

print("Data loaded.")

In [ ]:
def prepare_training_data_v5(results_df, features_df):
    training_rows = []
    feat_dict = features_df.set_index(['Season', 'TeamID']).to_dict('index')
    cols = ['SeedInt', 'OffEff', 'DefEff', 'Elo', 'AvgRank']
    
    for i, row in results_df.iterrows():
        s, w, l = row['Season'], row['WTeamID'], row['LTeamID']
        try:
            wf, lf = feat_dict[(s, w)], feat_dict[(s, l)]
            training_rows.append({**{f'{c}Diff': wf[c] - lf[c] for c in cols}, 'label': 1})
            training_rows.append({**{f'{c}Diff': lf[c] - wf[c] for c in cols}, 'label': 0})
        except KeyError: continue
    return pd.DataFrame(training_rows)

train_df = prepare_training_data_v5(tourney_results, team_features)
X = train_df.drop('label', axis=1)
y = train_df['label']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Training data scaled and ready.")

## 1. Out-of-Fold (OOF) Stacking

Generate predictions for the meta-learner.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_xgb, oof_lgb, oof_nn = np.zeros(len(X)), np.zeros(len(X)), np.zeros(len(X))

for tr_idx, val_idx in kf.split(X):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    X_tr_s, X_val_s = X_scaled[tr_idx], X_scaled[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    
    # XGBoost
    m_xgb = xgb.XGBClassifier(eval_metric='logloss', max_depth=4, learning_rate=0.05, n_estimators=150)
    m_xgb.fit(X_tr, y_tr)
    oof_xgb[val_idx] = m_xgb.predict_proba(X_val)[:, 1]
    
    # LightGBM
    m_lgb = lgb.LGBMClassifier(verbose=-1, num_leaves=16, learning_rate=0.05, n_estimators=150)
    m_lgb.fit(X_tr, y_tr)
    oof_lgb[val_idx] = m_lgb.predict_proba(X_val)[:, 1]
    
    # Neural Network
    m_nn = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=500, random_state=42)
    m_nn.fit(X_tr_s, y_tr)
    oof_nn[val_idx] = m_nn.predict_proba(X_val_s)[:, 1]

X_meta = pd.DataFrame({'xgb': oof_xgb, 'lgb': oof_lgb, 'nn': oof_nn})
meta_model = LogisticRegression()
meta_model.fit(X_meta, y)

print(f"Meta-learner trained. Ensemble Log Loss: {log_loss(y, meta_model.predict_proba(X_meta)[:, 1]):.4f}")

## 2. Final Prediction Pipeline

In [ ]:
# Pre-train final base models
f_xgb = xgb.XGBClassifier(eval_metric='logloss', max_depth=4, learning_rate=0.05, n_estimators=150).fit(X, y)
f_lgb = lgb.LGBMClassifier(verbose=-1, num_leaves=16, learning_rate=0.05, n_estimators=150).fit(X, y)
f_nn = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=500, random_state=42).fit(X_scaled, y)

def get_stacked_pred(season, t1, t2, feat_dict, models, scaler, meta):
    try:
        f1, f2 = feat_dict[(season, t1)], feat_dict[(season, t2)]
        cols = ['SeedInt', 'OffEff', 'DefEff', 'Elo', 'AvgRank']
        feat_row = pd.DataFrame([{f'{c}Diff': f1[c] - f2[c] for c in cols}])
        feat_s = scaler.transform(feat_row)
        
        p1 = models['xgb'].predict_proba(feat_row)[:, 1][0]
        p2 = models['lgb'].predict_proba(feat_row)[:, 1][0]
        p3 = models['nn'].predict_proba(feat_s)[:, 1][0]
        
        meta_features = pd.DataFrame({'xgb': [p1], 'lgb': [p2], 'nn': [p3]})
        return np.clip(meta.predict_proba(meta_features)[:, 1][0], 0.02, 0.98)
    except KeyError: return 0.5

submission = pd.read_csv(os.path.join(DATA_PATH, 'SampleSubmissionStage1.csv'))
feat_look = team_features.set_index(['Season', 'TeamID']).to_dict('index')
mods = {'xgb': f_xgb, 'lgb': f_lgb, 'nn': f_nn}

submission['Pred'] = submission.apply(lambda r: get_stacked_pred(int(r['ID'].split('_')[0]), int(r['ID'].split('_')[1]), int(r['ID'].split('_')[2]), feat_look, mods, scaler, meta_model), axis=1)
submission.to_csv('submission_v5.csv', index=False)
print("Submission v5 (Meta-Stacking) complete!")